In [1]:
import sys
sys.path.append("..")
import torch
from src.dataset import load_data, get_splits, HAM10000Dataset, train_transform, eval_transform
from src.evaluate import compute_class_weights, get_criterion, evaluate
from torch.utils.data import DataLoader

# reload data + splits (same as Day 2)
df, class_to_idx = load_data(data_dir="../data")
train_df, val_df, test_df = get_splits(df)
class_names = list(class_to_idx.keys())

val_loader = DataLoader(HAM10000Dataset(val_df, eval_transform), batch_size=32, shuffle=False)

# class weights
class_weights, train_counts = compute_class_weights(train_df, num_classes=len(class_to_idx))
print(train_counts)
print(class_weights)

# weighted loss
criterion = get_criterion(class_weights)

# sanity-check evaluate() runs end-to-end using a dummy model
class DummyModel(torch.nn.Module):
    def forward(self, x):
        return torch.randn(x.size(0), len(class_names))

dummy = DummyModel()
results = evaluate(dummy, val_loader, device=torch.device("cpu"), class_names=class_names)
print(results['balanced_accuracy'], results['macro_f1'])
print(results['report'])

label
0     229
1     360
2     769
3      81
4     779
5    4693
6      99
Name: count, dtype: int64
tensor([ 4.3731,  2.7817,  1.3022, 12.3633,  1.2855,  0.2134, 10.1154])
0.1037268969529251 0.08540830321136876
              precision    recall  f1-score   support

       akiec       0.02      0.12      0.04        49
         bcc       0.05      0.13      0.07        77
         bkl       0.10      0.13      0.11       165
          df       0.00      0.00      0.00        17
         mel       0.12      0.17      0.14       167
          nv       0.67      0.13      0.22      1006
        vasc       0.00      0.05      0.01        21

    accuracy                           0.13      1502
   macro avg       0.14      0.10      0.09      1502
weighted avg       0.47      0.13      0.18      1502

